# Tutorial 02b: Multi-Turn Conversations - Stateful Travel Assistant
## Microsoft Foundry Agent Service Edition
## Learning Objectives
By the end of this notebook, you will be able to:
- Understand what sessions are and why they are essential
- Create stateful conversations that remember context
- Manage conversation sessions with Azure AI
- Build natural, flowing interactions with a travel assistant
- Learn when to create new sessions vs. reuse existing ones

## Key Concepts

### What is a Session?
**Session (AgentSession)** is a conversation history that maintains context across multiple queries.

**Without a Session:**
```
User: "I want to go to Japan"
Agent: "Great! Japan has many attractions..."

User: "What's the weather like there?"
Agent: "Which location would you like weather info for?"  Forgot about Japan!
```

**With a Session:**
```
User: "I want to go to Japan"
Agent: "Great! Japan has many attractions..."

User: "What's the weather like there?"
Agent: "Let me check the weather in Japan for you..."  Remembers!
```

### Types of Sessions

1. **Automatic Sessions** (Stateless)
   - A new session is created for each `run()` call
   - No memory between queries
   - Good for: one-off questions

2. **Persistent Sessions** (Stateful)
   - The same session is reused across multiple calls
   - Maintains complete conversation history
   - Good for: multi-turn conversations

---


## Step 1: Setup and Imports


In [ ]:
import asyncio
import os
from random import randint, choice
from typing import Annotated

from agent_framework import Agent, AgentSession
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential, DefaultAzureCredential
from pydantic import Field
from dotenv import load_dotenv

load_dotenv(override=True)
print("✅ Imports successful!")


In [ ]:
import os
from agent_framework.azure import AzureOpenAIChatClient

mcp_server_uri = os.getenv("MCP_SERVER_URI")

project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model_deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

# NOTE: AzureAIAgentClient is an Async Context Manager.
# When using Agent with `async with`, the client is also entered/closed internally.
# Therefore, reusing a globally created AzureAIAgentClient across multiple cells/demos
# can easily cause "HTTP transport has already been closed" errors.
# → In the following demos, we use the approach of "creating a new client each time".
def create_foundry_chat_client(credential):
    return AzureAIAgentClient(
        credential=credential,
        project_endpoint=project_endpoint,
        model_deployment_name=model_deployment,
    )

# Reference: Use this if you want to use Azure OpenAI (Chat Completions)
# def create_azure_openai_chat_client():
#     return AzureOpenAIChatClient(
#         api_key=azure_openai_key,
#         endpoint=azure_openai_endpoint,
#         api_version=api_version,
#         deployment_name=azure_deployment,
#     )

print(f"MCP Server URI: {mcp_server_uri}")
print(f"Foundry Project Endpoint: {project_endpoint}")
print(f"Foundry Model Deployment: {model_deployment}")


## Setting Up a Tracer with OpenTelemetry
Using an OpenTelemetry tracer is helpful for debugging multi-agent scenarios.
Since `setup_observability` is not provided in this environment's Agent Framework,
we manually set up the OpenTelemetry `TracerProvider` (including Exporter/Processor) and enable instrumentation with `enable_instrumentation()`.
Here we use OTLP gRPC (e.g., Jaeger / OpenTelemetry Collector on port `4317`) as the trace destination.

Jaeger UI: [http://localhost:16686](http://localhost:16686)


In [ ]:
import os

from agent_framework.observability import configure_otel_providers, get_tracer

service_name = "agent_framework"
otlp_endpoint = os.getenv("OTEL_EXPORTER_OTLP_ENDPOINT", "http://localhost:4317")

# Configure via environment variables (Agent Framework reads standard OTEL_* vars)
os.environ.setdefault("OTEL_SERVICE_NAME", service_name)
os.environ.setdefault("OTEL_EXPORTER_OTLP_ENDPOINT", otlp_endpoint)
os.environ.setdefault("OTEL_EXPORTER_OTLP_PROTOCOL", "grpc")
os.environ.setdefault("ENABLE_INSTRUMENTATION", "true")

# Set enable_sensitive_data=True to enable collection of sensitive data (OpenAI IN/OUT data)
configure_otel_providers(enable_sensitive_data=True)

tracer = get_tracer()
print(f"Observability configured: service={service_name} otlp={otlp_endpoint}")


## Step 2: Define Travel Tools

Let's create the tools that the agent will use throughout the conversation.


In [ ]:
def get_weather(
    location: Annotated[str, Field(description="City or country name")],
) -> str:
    """Returns the current weather for the specified location."""
    conditions = ["Sunny", "Partly Cloudy", "Cloudy", "Rainy", "Windy"]
    temp = randint(15, 32)
    condition = choice(conditions)
    return f"Weather in {location}: {condition}, {temp}°C"


def get_attractions(
    location: Annotated[str, Field(description="City or country name")],
) -> str:
    """Returns the main tourist attractions for the destination."""
    # Mock data for tourist attractions
    attractions = {
        "Japan": "Tokyo Tower, Mount Fuji, Kyoto Temples, Osaka Castle",
        "Paris": "Eiffel Tower, Louvre Museum, Notre-Dame Cathedral, Arc de Triomphe",
        "London": "Big Ben, Tower of London, British Museum, Buckingham Palace",
        "Barcelona": "Sagrada Familia, Park Guell, La Rambla, Gothic Quarter",
    }

    for city, attrs in attractions.items():
        if city.lower() in location.lower():
            return f"Top attractions in {location}: {attrs}"

    return f"Popular attractions in {location}: Historical sites, Museums, Local markets, Parks"


def estimate_budget(
    destination: Annotated[str, Field(description="Destination city or country")],
    days: Annotated[int, Field(description="Number of days")],
) -> str:
    """Estimates the travel budget based on destination and number of days."""
    daily_costs = {
        "Japan": 150,
        "Paris": 180,
        "London": 200,
        "Barcelona": 120,
        "Thailand": 60,
    }

    cost_per_day = daily_costs.get(destination, 100)
    total = cost_per_day * days

    return f"Estimated budget for {days} days in {destination}: ${total} (${cost_per_day} per day)"


print("✅ Tool definitions complete")


## Step 3: The Problem - No Memory (Stateless)

Let's see what happens without sessions - each query is independent.


In [ ]:
async def example_without_sessions():
    """
    Demonstrates the problem: the agent doesn't remember previous context.
    A new session is automatically created for each run() call.
    """
    print("=== Without Session (No Memory) ===")
    print("Each query is independent - the agent forgets previous conversations\n")

    async with (
        DefaultAzureCredential() as credential,
        create_foundry_chat_client(credential) as foundry_client,
        Agent(
            client=foundry_client,
            name="StatelessTravelAgent",
            instructions="You are a helpful travel assistant.",
            tools=[get_weather, get_attractions, estimate_budget],
        ) as agent,
    ):
        # First query
        query1 = "I'm planning a trip to Japan next month."
        print(f"User: {query1}")
        response1 = await agent.run(query1)
        print(f"Agent: {response1.text}\n")

        # Second query - expecting the agent to remember Japan
        query2 = "What's the weather like there?"
        print(f"User: {query2}")
        response2 = await agent.run(query2)
        print(f"Agent: {response2.text}\n")

        print("❌ Notice: The agent doesn't know where \"there\" is!")
        print("   Each run() created a separate session.\n")


await example_without_sessions()


## Step 4: The Solution - Persistent Sessions

Now let's use **persistent sessions** to maintain conversation context.


In [ ]:
async def example_with_session():
    """
    This shows the solution: using the same session maintains context.
    """
    print("=== With Persistent Session (With Memory) ===")
    print("The same session maintains context across multiple queries\n")

    async with (
        DefaultAzureCredential() as credential,
        create_foundry_chat_client(credential) as foundry_client,
        Agent(
            client=foundry_client,
            name="StatefulTravelAgent",
            instructions="You are a helpful travel assistant.",
            tools=[get_weather, get_attractions, estimate_budget],
        ) as agent,
    ):
        # Create a new session that we'll reuse
        session = agent.create_session()

        # First query
        query1 = "I'm planning a trip to Japan next month."
        print(f"User: {query1}")
        response1 = await agent.run(query1, session=session)
        print(f"Agent: {response1.text}\n")

        # Second query - agent remembers Japan!
        query2 = "What's the weather like there?"
        print(f"User: {query2}")
        response2 = await agent.run(query2, session=session)
        print(f"Agent: {response2.text}\n")

        # Third query - agent remembers the whole conversation
        query3 = "What are the top attractions to visit?"
        print(f"User: {query3}")
        response3 = await agent.run(query3, session=session)
        print(f"Agent: {response3.text}\n")

        print("✅ Success: The agent remembers the entire conversation!")
        print(f"   Session ID: {session.service_session_id}\n")

await example_with_session()


## Step 5: A Natural Multi-Turn Planning Conversation

Let's have a realistic travel planning conversation with multiple interaction turns.


In [ ]:
async def planning_conversation():
    """
    A natural conversation where the agent helps plan a trip.
    """
    print("=== Natural Travel Planning Conversation ===")
    print("Context-aware multi-turn dialogue\n")

    async with (
        DefaultAzureCredential() as credential,
        create_foundry_chat_client(credential) as foundry_client,
        Agent(
            client=foundry_client,
            name="TravelPlanningAgent",
            instructions="""
            You are a travel planning expert. Help the user plan their trip by:
            - Asking clarifying questions as needed
            - Remembering all details the user shares
            - Using tools to provide accurate information
            - Being enthusiastic and helpful
            """,
            tools=[get_weather, get_attractions, estimate_budget],
        ) as agent,
    ):
        session = agent.create_session()

        # Conversation flow
        queries = [
            "I have 2 weeks off in December and want to go somewhere warm.",
            "I'm considering Thailand or Barcelona. Which would you recommend?",
            "I've decided on Barcelona! What's the weather like in December?",
            "How much budget should I plan for 7 days?",
            "That works for me! What are the must-see attractions?",
        ]

        for i, query in enumerate(queries, 1):
            print(f"{'='*60}")
            print(f"Turn {i}")
            print(f"{'='*60}")
            print(f"User: {query}\n")

            response = await agent.run(query, session=session)
            print(f"Agent: {response.text}\n")

        print("✅ Conversation completed with full context maintained!")
        print(f"   Session ID: {session.service_session_id}\n")

await planning_conversation()


## Retrieving Thread Contents (Messages) via Foundry Agent Service SDK


In [ ]:
from azure.ai.agents import AgentsClient
from azure.identity import DefaultAzureCredential as SyncDefaultAzureCredential
from azure.ai.agents.models import ListSortOrder

thread_id = "thread_xxx"

order = ListSortOrder.ASCENDING

agents_client = AgentsClient(
    endpoint=project_endpoint,
    credential=SyncDefaultAzureCredential(),
)

messages = agents_client.messages.list(thread_id=thread_id, order=order)
print(f"Message history for Thread ID: {thread_id}:")
count = 0
for msg in messages:
    count += 1
    role = getattr(msg, "role", None) or "unknown"
    if getattr(msg, "text_messages", None):
        text = "\n".join(t.text.value for t in msg.text_messages)
    else:
        text = str(msg)

    # Highlight the start of each role (separator line + heading)
    header = f"[MessageRole.{role}]"
    sep = "=" * max(24, len(header) + 8)
    print(sep)
    print(f"{header}")
    print(sep)
    print(text)
    print()

if count == 0:
    print("(No messages found)")


## Step 6: Multiple Conversations with Different Sessions

You can manage multiple conversations simultaneously using different sessions.


In [ ]:
async def multiple_conversations():
    """
    Manage multiple separate conversations with different sessions.
    """
    print("=== Multiple Conversations (Different Sessions) ===")
    print("Each session maintains its own context\n")

    async with (
        DefaultAzureCredential() as credential,
        create_foundry_chat_client(credential) as foundry_client,
        Agent(
            client=foundry_client,
            name="MultiConversationAgent",
            instructions="You are a helpful travel assistant.",
            tools=[get_weather, get_attractions],
        ) as agent,
    ):
        # Create two separate sessions for two different users/conversations
        alice_session = agent.create_session()
        bob_session = agent.create_session()

        # Alice's conversation about Paris
        print("🧑 Alice's Conversation:")
        alice_q1 = "I'm interested in visiting Paris."
        print(f"Alice: {alice_q1}")
        response = await agent.run(alice_q1, session=alice_session)
        print(f"Agent: {response.text[:100]}...\n")

        # Bob's conversation about London
        print("👨 Bob's Conversation:")
        bob_q1 = "I'm thinking about London for my next trip."
        print(f"Bob: {bob_q1}")
        response = await agent.run(bob_q1, session=bob_session)
        print(f"Agent: {response.text[:100]}...\n")

        # Continue Alice's conversation
        print("🧑 Alice continues:")
        alice_q2 = "What's the weather like there?"
        print(f"Alice: {alice_q2}")
        response = await agent.run(alice_q2, session=alice_session)
        print(f"Agent: {response.text}\n")

        # Continue Bob's conversation
        print("👨 Bob continues:")
        bob_q2 = "What attractions should I visit there?"
        print(f"Bob: {bob_q2}")
        response = await agent.run(bob_q2, session=bob_session)
        print(f"Agent: {response.text}\n")

        print("✅ Both conversations maintained separate context!")
        print(f"   Alice's Session: {alice_session.service_session_id}")
        print(f"   Bob's Session: {bob_session.service_session_id}\n")

await multiple_conversations()


## Step 7: Resuming Conversations (Two Approaches)

How to "resume a conversation later" depends on whether the **client can manage sessions on the service side**.

### (A) Resume with Foundry (Azure AI) Service-Managed Sessions
- `session.service_session_id` is an ID that points to a **conversation session stored on the service side (Foundry)**.
- If you save this ID to a database, you can resume later with `AgentSession(service_session_id=...)`.
- The `session_id` in `AzureAIAgentClient(..., session_id=...)` means "the default session ID that this client uses".

### (B) Resume with AzureOpenAIChatClient Local History
- The Chat Completions-based `AzureOpenAIChatClient` typically does not rely on **service-managed sessions like Foundry's session_id**.
- In that case, save the result of `session.to_dict()` (local conversation history) and restore it later with `AgentSession.from_dict(...)` to resume.

The following cells demonstrate (A) and (B) respectively.


In [ ]:
async def resume_conversation_foundry():
    """
    (A) Example of resuming with a Foundry (Azure AI) service-managed session.
    - Save session.service_session_id and restore later with AgentSession(service_session_id=...).
    """
    print("=== (A) Foundry: Resume Conversation with Service-Managed Session ===")

    saved_session_id: str | None = None

    # Part 1: Start a conversation and save the service-managed session ID
    print("\n--- Part 1: Start a conversation in Foundry and save the session ID ---")
    try:
        async with (
            DefaultAzureCredential() as credential,
            create_foundry_chat_client(credential) as foundry_client,
            Agent(
                client=foundry_client,
                name="ResumableFoundryAgent",
                instructions="You are a helpful travel assistant.",
                tools=[get_weather, estimate_budget],
            ) as agent,
        ):
            session = agent.create_session()
            query = "I want to visit Barcelona for 5 days. What would the budget be?"
            print(f"User: {query}")
            response = await agent.run(query, session=session)
            print(f"Agent: {response.text}\n")

            saved_session_id = session.service_session_id
            print(f"✅ Session ID saved: {saved_session_id}")
            print("   (In a real app, you would save this to a database)\n")
    except Exception as ex:
        print("⚠️ Could not run the Foundry session resume demo.")
        print(f"Reason: {ex}\n")
        print("Hint: Check your Foundry/Azure AI environment variables and permissions (RBAC).\n")
        return

    if not saved_session_id:
        print("⚠️ service_session_id could not be obtained.")
        print("This client may not return service-managed sessions.\n")
        return

    # Part 2: Resume later using the saved session ID
    print("--- Part 2: Resume conversation from saved session ID ---")
    try:
        async with (
            DefaultAzureCredential() as credential,
            create_foundry_chat_client(credential) as foundry_client,
            Agent(
                client=foundry_client,
                name="ResumableFoundryAgent",
                instructions="You are a helpful travel assistant.",
                tools=[get_weather, estimate_budget],
            ) as agent,
        ):
            session = AgentSession(service_session_id=saved_session_id)
            query = "What's the weather like there?"
            print(f"User: {query}")
            response = await agent.run(query, session=session)
            print(f"Agent: {response.text}\n")

            print("✅ Successfully resumed conversation from saved session ID!")
            print("   The agent remembered Barcelona from the previous session.\n")
    except Exception as ex:
        print("⚠️ Failed to execute the resume.")
        print(f"Reason: {ex}\n")



await resume_conversation_foundry()


## Step 8: Inspecting Session Messages

You can examine the conversation history stored in the session.


In [ ]:
async def inspect_session():
    """
    Inspect the messages and history within a session.
    Note: With Azure AI service-managed sessions, messages are stored in Azure.
    """
    print("=== Inspecting Session Messages ===")

    async with (
        DefaultAzureCredential() as credential,
        create_foundry_chat_client(credential) as foundry_client,
        Agent(
            client=foundry_client,
            name="InspectableAgent",
            instructions="You are a helpful travel assistant.",
            tools=[get_weather],
        ) as agent,
    ):
        session = agent.create_session()

        # Have a short conversation
        queries = ["What's the weather in Tokyo?", "How about London?", "Which has better weather?"]

        print("Conversing...\n")
        for i, query in enumerate(queries, 1):
            print(f"Query {i}: {query}")
            response = await agent.run(query, session=session)
            print(f"Response: {response.text[:100]}...\n")

        # Inspect the session
        print("=" * 60)
        print(f"Session ID: {session.service_session_id}")
        print(f"Service-Managed Session: {session.service_session_id is not None}")
        print(f"Session State: {session.state}")
        print("=" * 60)
        print("\nNote: In Azure AI, messages are stored on the service.")
        print("The session ID is used to access conversation history in Azure.")

await inspect_session()


## 🗑 Agent Lifecycle Management in Microsoft Foundry

### Automatic Agent Cleanup

**Important:** In Microsoft Foundry, depending on how you create agents via the SDK, **agents are automatically deleted after execution completes.**

#### How It Works

```python
async with (
    AzureCliCredential() as credential,
    Agent(
        client=AzureAIAgentClient(credential=credential),
        name="TravelAgent",
        instructions="You are a travel assistant.",
    ) as agent,  # ← Agent is created here
):
    # Agent exists in Azure within this block
    response = await agent.run("Hello")
    
# ← Agent is deleted from Azure (when context exits)
```

**What happens:**
1. **Agent Creation** - When entering the `async with` block, the agent is created in Microsoft Foundry
2. **Agent Usage** - You can execute queries, create sessions, and use tools
3. **Agent Deletion** - When exiting the block, the agent definition is deleted from Azure
4. **Sessions Persist** - Session data remains in Azure (only the agent definition is deleted)

### What Gets Deleted vs. What Persists

| Resource | After Context Exit | Reason |
|----------|-------------------|--------|
| **Agent Definition** | ❌ Deleted | Reduces clutter and saves costs |
| **Agent Name** | ❌ Deleted | Agent resource is deleted |
| **Agent Instructions** | ❌ Deleted | Part of agent definition |
| **Agent Tools** | ❌ Deleted | Part of agent definition |
| **Session ID** | ✅ Persists | Conversation history is retained |
| **Session Messages** | ✅ Persists | Data is retained |
| **Model Deployment** | ✅ Persists | Shared resource |

### Benefits of Automatic Deletion

#### 1. **Cost Efficiency** 
```python
# Traditional approach (agents persist)
agent_id = "agent-123"  # Agent stays in Azure, potentially causing governance and maintenance issues

# Microsoft Foundry approach (ephemeral agents)
async with Agent(...) as agent:
    # Agent exists only when needed
    await agent.run(query)
# Agent deleted
```

**Benefit:** You only pay for compute while running, not for idle agent storage

#### 2. **Resource Management** 🧹
```python
# Problem with persistent agents:
# - Accumulation of stale/test agents
# - Namespace pollution
# - Manual cleanup required

# Microsoft Foundry solution:
async with Agent(name="TestAgent") as agent:
    # Create and test
    pass
# Automatic cleanup - no leftover test agents!
```

**Benefit:** No "agent sprawl" - your Azure workspace stays clean

#### 3. **Security and Compliance** 
```python
# Sensitive agent with PII access
async with Agent(
    name="CustomerDataAgent",
    instructions="Access customer PII for support purposes",
) as agent:
    # Agent exists temporarily
    response = await agent.run("Retrieve user information")
# Agent definition deleted, reducing attack surface
```

**Benefit:** Short-lived credentials, reduced security risk

#### 4. **Version Control and GitOps** 
```python
# Agent definition in code
async with Agent(
    name="V2TravelAgent",  # Version in code
    instructions="Updated instructions..."  # Source controlled
) as agent:
    await agent.run(query)
```

**Benefit:** Agent behavior is defined in code, not in Azure state

### Drawbacks and Trade-offs

| Drawback | Mitigation |
|----------|------------|
| **Agent recreated on every execution** | Overhead is minimal (~100-300ms) |
| **Cannot view agent in portal** | View sessions/messages instead |
| **Requires code for definition** | Better suited for version control |
| **No long-running agents** | Use sessions for continuity |

### Alternative: Persistent Agents 

You can also make agents persistent:

```python
agent = create_agent(name="MyAgent")  # Agent remains on the platform
agent_id = agent.id  # "agent-abc123"

# Later (different session)
agent = get_agent(agent_id)  # Retrieve existing agent
response = agent.run("Hello")  # Use persistent agent
```

**When persistent agents make sense:**
- Agents with complex, evolving instructions
- Agents shared across multiple services
- Agents that learn/adapt over time
- Long-running background agents

**Microsoft Foundry's Philosophy:**
- Agents are **code** (in the repository)
- Sessions are **data** (in Azure)
- Separation of compute (ephemeral) and state (persistent)

### Best Practices for Microsoft Foundry

#### 1. **Define Agents as Code**
```python
# Good example: Agent configuration in code
def create_travel_agent(credential):
    return Agent(
        client=AzureAIAgentClient(credential=credential),
        name="TravelAgent",
        instructions="You are a helpful travel assistant...",
        tools=[get_weather, get_attractions]
    )

# Usage
async with create_travel_agent(credential) as agent:
    response = await agent.run(query, session=session)
```

#### 2. **Persist Session IDs, Not Agent IDs**
```python
# Save to database
session_data = {
    "user_id": user.id,
    "session_id": session.service_session_id,  # ✅ Save this
    # "agent_id": agent.id  # ❌ Don't do this (agent will be deleted)
}

# Restore later
async with create_travel_agent(credential) as agent:  # New agent
    session = AgentSession(service_session_id=session_data["session_id"])  # Same session
    response = await agent.run(query, session=session)
```

#### 3. **Use Context Managers**
```python
# ✅ Good: Automatic cleanup
async with Agent(...) as agent:
    await agent.run(query)

# ❌ Avoid: Manual management
agent = Agent(...)
await agent.run(query)
# Agent may not be properly cleaned up
```

### Comparison Table: Azure AI vs. Traditional

| Aspect | Microsoft Foundry | Traditional Persistent |
|--------|-------------------|------------------------|
| Agent Lifetime | Ephemeral (seconds) | Persistent (days/months) |
| Storage Cost | No agent cost | Ongoing agent cost |
| Cleanup | Automatic | Manual required |
| Version Control | In code (Git) | In platform (API) |
| Scaling | Unlimited agents | Limited by quota |
| Session Persistence | ✅ Yes | ✅ Yes |
| Agent Evolution | Code updates | Platform updates |
| Multi-tenancy | Isolated per execution | Shared agents |


## Understanding the Session Lifecycle

### Creating and Managing Sessions

```python
# Option 1: Let sessions be created and managed automatically
response = await agent.run(query)  # New session each time

# Option 2: Create a session and reuse it (recommended)
session = agent.create_session()
response = await agent.run(query, session=session)  # Same session

# Option 3: Resume from a saved session ID
session = AgentSession(service_session_id=saved_id)
response = await agent.run(query, session=session)
```

### Session Storage in Azure

- Sessions are **stored in Microsoft Foundry**
- Session IDs are **persistent** across sessions
- You can **resume conversations** at any time
- Azure manages **cleanup and retention**

### When to Use Sessions

| Use Case | Session Strategy |
|----------|------------------|
| One-off questions | No session (automatic) |
| Chat conversations | Single persistent session |
| Multi-user apps | One session per user |
| Session-based | New session per session |
| Long-term assistant | Save and resume session ID |


## 🔬 Practical Example: Agent Lifecycle Demo

Let's see ephemeral agent behavior in action:


In [ ]:
async def agent_lifecycle_demo():
    """
    Demonstrates ephemeral agent behavior with persistent sessions.
    Key points:
    - Agent definitions are deleted when exiting the `async with` block (ephemeral)
    - If the session is service-managed (Foundry), a `session.service_session_id` is issued and can be resumed in another session
    """
    print("=== Agent Lifecycle Demo ===\n")

    if not project_endpoint or not model_deployment:
        print("⚠️ Missing Foundry configuration.")
        print("   Please set AZURE_AI_PROJECT_ENDPOINT / AZURE_AI_MODEL_DEPLOYMENT_NAME.\n")
        return

    saved_session_id: str | None = None

    # Session 1: Create agent → Converse → Agent deleted
    print("--- Session 1: Creating agent and session ---")
    async with (
        DefaultAzureCredential() as credential,
        create_foundry_chat_client(credential).as_agent(
            name="EphemeralAgent_Session1",
            instructions="You are a helpful assistant. Remember the information the user tells you.",
            tools=[get_weather],
        ) as agent,
    ):
        print(f"✅ Agent created: {agent.name}")

        session = agent.create_session()

        query1 = "My favorite city is Barcelona. Please remember that."
        print(f"User: {query1}")
        response1 = await agent.run(query1, session=session)
        print(f"Agent: {response1.text}\n")

        saved_session_id = session.service_session_id
        print(f"✅ Session ID saved: {saved_session_id}")
        print("   (In a real app, you would save this to a database)\n")

    print("\n🗑  Agent 'EphemeralAgent_Session1' has been deleted from Azure")
    print("    (Automatically deleted when the 'async with' block exits)")
    print("📦 Session data remains in Microsoft Foundry\n")

    if not saved_session_id:
        print("⚠️ service_session_id remains None.")
        print("   This means the session was not created as a service-managed session.")
        print("   Possible causes:")
        print("   - Incorrect Foundry connection info/permissions")
        print("   - Using a client that does not support service-managed sessions\n")
        return

    # Session 2: New agent + same session = conversation continues
    print("--- Session 2: New agent, same session ---")
    async with (
        DefaultAzureCredential() as credential,
        create_foundry_chat_client(credential).as_agent(
            name="EphemeralAgent_Session2",
            instructions="You are a helpful assistant. Remember the information the user tells you.",
            tools=[get_weather],
        ) as agent,
    ):
        print(f"✅ New agent created: {agent.name}")
        print(f"✅ Session resumed: {saved_session_id}\n")

        # Resume the same session (linked to the service-side conversation history)
        session = AgentSession(service_session_id=saved_session_id)

        query2 = "What was my favorite city?"
        print(f"User: {query2}")
        response2 = await agent.run(query2, session=session)
        print(f"Agent: {response2.text}\n")

        query3 = "What's the weather like in that city?"
        print(f"User: {query3}")
        response3 = await agent.run(query3, session=session)
        print(f"Agent: {response3.text}\n")

    print("\n🗑  Agent 'EphemeralAgent_Session2' has been deleted from Azure\n")

    print("=" * 60)
    print("Key Insights:")
    print("=" * 60)
    print("1. Agents are ephemeral - created and deleted with each session")
    print("2. Sessions are persistent - conversation history is retained")
    print("3. Different agents can continue the same conversation")
    print("4. Session IDs are the key to conversation continuity")
    print("5. Agent definitions live in code, not in Azure")
    print("=" * 60)


await agent_lifecycle_demo()


## Resource Scope / Lifetime
The `async with (...)` above may look unusual, but what it actually does is simply **generate → destroy multiple "cleanup-equipped resources" reliably in nested fashion**. The key point is that the **scope (effective range)** and **lifetime (when created and when closed/destroyed)** are strictly determined by `async with`.

Within this single block, there are at least the following items with "different lifetimes":

- `credential` (authentication handle)
  - Lifetime: Valid only within the `async with DefaultAzureCredential()` block
  - On exit: `credential` is closed, and internal HTTP connections may also be closed

- The client returned by `create_foundry_chat_client(credential)` (e.g., `AzureAIAgentClient`)
  - Lifetime: Here it's only used inside `.as_agent(...)` (with this syntax, it's not bound to a variable, so it's "temporary")
  - On exit: `.as_agent(...).__aexit__()` may close the client/transport (implementation-dependent)

- `agent` (the "ephemeral agent" created on the Foundry side)
  - Lifetime: Only within the `async with ... as agent:` block
  - On exit: **The agent definition on the service side is deleted** the moment the block is exited (= ephemeral)

- `session` (conversation session)
  - Lifetime: Two possible types
    - **Service-managed session**: A `session.service_session_id` is issued, and you can save the ID and restore it later (session persists even when the agent is deleted)
    - Otherwise: Only held locally, and won't persist unless you save the state or the process ends

In other words, this `async with` enforces the design of "agents are ephemeral, sessions are persistent (potentially)".

## Why It Looks Unusual (Why It's Hard to Read)
- A single `async with` packs **3 levels of lifecycles** (credential / client / agent)
- Furthermore, `.as_agent(...)` is a two-stage construct that "generates a context manager for the agent from the client", making the generation and destruction boundaries hard to see


## Key Takeaways

### Benefits of Sessions
1. **Context Awareness** - The agent remembers previous interactions
2. **Natural Conversations** - Users don't need to repeat information
3. **Better UX** - Feels like talking to a human
4. **Stateful Interactions** - Builds on previous responses

### Best Practices
1. **Use sessions for conversations** - Always use them for multi-turn dialogues
2. **Save session IDs** - Store them in your user database
3. **One session per conversation** - Don't mix different topics
4. **Clean up old sessions** - Delete when conversations end (Azure helps with this)

### Common Patterns

**Web Chat Application:**
```python
# Save session_id to user session
if not session_store.get('session_id'):
    session = agent.create_session()
    session_store['session_id'] = session.service_session_id
else:
    session = AgentSession(service_session_id=session_store['session_id'])
```

**Customer Support:**
```python
# One session per support ticket
session_id = ticket.get_session_id()
session = AgentSession(service_session_id=session_id)
```


## Practice Exercises

1. **Create a Booking Conversation** - Plan a complete trip from start to booking
2. **Destination Comparison** - Have the agent remember and compare multiple cities
3. **Build a Preference Profile** - Have the agent learn user preferences through conversation
4. **Implement Session Management** - Save/load session IDs from a dictionary


In [ ]:
# Exercise: Create a conversation where the agent helps compare 3 destinations
# and remembers all the details discussed about each one

async def compare_destinations_exercise():
    """
    Your turn! Create a conversation that:
    1. Discusses Paris (weather, attractions, budget)
    2. Discusses Tokyo (same info)
    3. Discusses Barcelona (same info)
    4. Asks agent to compare all three and recommend
    
    The agent should remember ALL details from the conversation!
    """
    # Your code here
    pass

# Uncomment to test your solution
# await compare_destinations_exercise()


## Next Steps

Great! You've learned how to create stateful, context-aware conversations.

However, agents still have limitations:
- ❌ Can't remember user preferences across sessions
- ❌ Can't store long-term knowledge about users
- ❌ Session conversation history can become very long

In **Tutorial 03: Context and Memory**, you will learn:
- How to add persistent memory to agents
- How to save user preferences and profiles
- How to use context providers for smarter agents
- How to manage conversation context efficiently

---

### Quick Reference

**Create a Persistent Session:**
```python
session = agent.create_session()
response = await agent.run(query, session=session)
```

**Save and Resume:**
```python
# Save
session_id = session.service_session_id

# Resume
session = AgentSession(service_session_id=session_id)
```

**Multiple Conversations:**
```python
alice_session = agent.create_session()
bob_session = agent.create_session()
```
